# VideoSeek - Module 2: Query-Adaptive Perception & Dynamic FPS Focus
Notebook này dùng để kiểm thử trên **Google Colab** cho **Module 2: Query-Adaptive Perception & Dynamic FPS Focus** của dự án `VideoSeek`:
1. **Cell 1:** Clone repository `videoseek` và cài đặt các thư viện cần thiết (`decord`, `litellm`, `pillow`, `pyyaml`, `yt-dlp`, `openai-whisper`, `opencv-python-headless`, `pytest`).
2. **Cell 2:** Thiết lập biến môi trường kết nối LLM/VLM API (`OPENAI_API_KEY`, `OPENAI_API_BASE`, `OPENAI_API_VERSION`).
3. **Cell 3:** Chạy bộ **Unit Tests** tự động kiểm tra `QueryAnalyzer`, `Dynamic FPS Focus` (`fps=1, 4, 8`), và `Hybrid Anomaly Overview Sampling`.
4. **Cell 4:** Tạo **Synthetic Video** kiểm thử bằng OpenCV (video dài 120s/1200s có chèn dòng chữ màu đỏ `"BI MAT"` xuất hiện chớp nhoáng trong 3 giây) hoặc tải video thực tế.
5. **Cell 5:** Khởi chạy `VideoSeekAgent` với câu hỏi kiểm thử sự kiện phi cốt truyện, in chi tiết `QueryProfile`, lịch sử `Thought -> Action -> Observation`, và kiểm chứng Agent gọi `focus` với `fps=8`.

## Cell 1: Clone Repository & Cài đặt Thư viện trên Google Colab

In [8]:
import os
import sys

# 1. Clone repo nếu đang chạy trên Google Colab và chưa có thư mục videoseek
if not os.path.exists("videoseek"):
    !git clone https://github.com/nhatsonle/videoseek.git

# Di chuyển vào thư mục gốc của repo videoseek
if os.path.exists("videoseek") and os.path.isdir("videoseek") and os.path.exists("videoseek/config"):
    os.chdir("videoseek")

# Luôn cập nhật code mới nhất từ nhánh main
!git pull origin main

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Current working directory:", os.getcwd())

# 2. Cài đặt đầy đủ các thư viện yêu cầu
!pip install -q decord litellm pillow pyyaml yt-dlp openai-whisper opencv-python-headless pytest

Current working directory: /content/videoseek


## Cell 2: Cấu hình Biến Môi trường API (`OPENAI_API_KEY`, `OPENAI_API_BASE`, `OPENAI_API_VERSION`)

In [ ]:
import os
from config import general_config, prompts_config

# Tự động đọc từ Google Colab Secrets (nếu có), hoặc biến môi trường, hoặc điền trực tiếp
def _get_secret_or_env(key: str, default: str = "") -> str:
    val = os.getenv(key)
    if val:
        return val
    try:
        from google.colab import userdata
        secret_val = userdata.get(key)
        if secret_val:
            return secret_val
    except Exception:
        pass
    return default

os.environ["OPENAI_API_KEY"] = _get_secret_or_env("OPENAI_API_KEY", "YOUR_OPENAI_API_KEY")
os.environ["OPENAI_API_BASE"] = _get_secret_or_env("OPENAI_API_BASE", "https://hub.smilesoftware.org/v1")
os.environ["OPENAI_API_VERSION"] = _get_secret_or_env("OPENAI_API_VERSION", "")

# Điền trực tiếp tên model bạn muốn sử dụng tại đây:
MODEL_NAME = "gpt-6.1-terra"  # Có thể đổi thành "openai/gpt-6.1-terra", "gpt-5.6-terra", "gpt-4o", v.v.

# Chuẩn bị runtime config cho VideoSeekAgent
runtime_config = {**general_config, **prompts_config}
runtime_config["api_key"] = os.environ["OPENAI_API_KEY"]
runtime_config["api_base"] = os.environ["OPENAI_API_BASE"]
runtime_config["api_version"] = os.environ["OPENAI_API_VERSION"] or None
runtime_config["model_name"] = _get_secret_or_env("VIDEOSEEK_MODEL", MODEL_NAME)

print("Configured Model:", runtime_config["model_name"])
print("Adaptive FPS Enabled:", runtime_config.get("adaptive_fps_enabled"))
print("Anomaly Overview Enabled:", runtime_config.get("anomaly_overview_enabled"))
print("Max Focus Frames Cap:", runtime_config.get("max_focus_frames_cap"))

## Cell 3: Chạy Bộ Unit Tests Tự Động Kiểm Tra Module 2 (`unittest` / `pytest`)
Kiểm tra toàn diện:
1. `QueryAnalyzer`: Phân loại câu hỏi qua 1 LLM JSON Router duy nhất (`reasoning_effort="low"`, `return_json=True`) và cơ chế fallback an toàn khi lỗi API/JSON.
2. `focus_tool` & `execute_focus`: Schema bắt buộc có `fps`, tính toán số frame theo `duration_window * fps`, giới hạn bởi `max_focus_frames_cap=24`, xử lý an toàn `start_time >= end_time`, và format timestamp `:.2f` khi `fps >= 4`.
3. `execute_overview` & `_sample_hybrid_anomaly_frames`: Lấy mẫu lai 75% Uniform + 25% Anomaly Peak (tổng số frame chia hết cho 8) và bắt trúng sự kiện màu đỏ chớp nhoáng.

In [7]:
import os
import sys
import importlib
import importlib.util
import unittest

# Đảm bảo đang đứng ở thư mục gốc của videoseek và kéo bản cập nhật mới nhất từ GitHub
if not os.path.exists("tests/test_module2.py"):
    if os.path.exists("videoseek") and os.path.isdir("videoseek"):
        os.chdir("videoseek")
    !git pull origin main

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Reload lại các module của videoseek nếu đã được import từ bản cũ trước khi git pull
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith(("videoseek", "config")):
        importlib.reload(sys.modules[mod_name])

# Nạp trực tiếp file tests/test_module2.py (tránh xung đột tên gói 'tests' có sẵn trên Google Colab)
test_file_path = os.path.abspath("tests/test_module2.py")
spec = importlib.util.spec_from_file_location("test_module2", test_file_path)
test_module2 = importlib.util.module_from_spec(spec)
sys.modules["test_module2"] = test_module2
spec.loader.exec_module(test_module2)

TestModule2AdaptivePerception = test_module2.TestModule2AdaptivePerception

# Chạy toàn bộ 7 kịch bản Unit Tests từ tests/test_module2.py
suite = unittest.TestLoader().loadTestsFromTestCase(TestModule2AdaptivePerception)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful(), "Unit tests failed!"
print("\n[PASS] Toàn bộ 7/7 Unit Tests của Module 2 đã vượt qua thành công!")

ModuleNotFoundError: No module named 'tests.test_module2'

## Cell 4: Tạo Video Kiểm Thử Giả Lập (Synthetic Video bằng OpenCV)
Tạo một video kiểm thử (mặc định `120s`, có thể đổi `SYNTHETIC_DURATION_SEC = 1200` cho video 20 phút) gồm các cảnh chuyển động thông thường, nhưng tại khoảng **`44.0s - 47.0s`** (3 giây) xuất hiện đột ngột dòng chữ màu đỏ nổi bật **`"BI MAT"`** không có trong phụ đề `.srt`.

In [ ]:
import cv2
import numpy as np

SYNTHETIC_VIDEO_PATH = "synthetic_anomaly_test.mp4"
SYNTHETIC_SRT_PATH = "synthetic_anomaly_test.srt"
SYNTHETIC_DURATION_SEC = 120  # Có thể chỉnh thành 1200 để test video 20 phút
SYNTHETIC_FPS = 10            # 10 FPS giúp tạo video nhanh trên Colab mà vẫn đủ kiểm thử focus 8 FPS
ANOMALY_START_SEC = 44.0
ANOMALY_END_SEC = 47.0


def create_synthetic_test_video(
    video_path: str = SYNTHETIC_VIDEO_PATH,
    srt_path: str = SYNTHETIC_SRT_PATH,
    duration_sec: int = SYNTHETIC_DURATION_SEC,
    fps: int = SYNTHETIC_FPS,
    anomaly_start: float = ANOMALY_START_SEC,
    anomaly_end: float = ANOMALY_END_SEC,
):
    width, height = 640, 360
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(video_path, fourcc, fps, (width, height))
    total_frames = int(duration_sec * fps)

    for idx in range(total_frames):
        t = idx / float(fps)
        # Nền xám xanh dịu có hình tròn trắng di chuyển chậm mô phỏng mạch video bình thường
        frame = np.full((height, width, 3), (45, 55, 65), dtype=np.uint8)
        cx = int((t / duration_sec) * (width - 100)) + 50
        cy = height // 2
        cv2.circle(frame, (cx, cy), 30, (200, 200, 200), -1)
        cv2.putText(
            frame,
            f"Time: {t:.1f}s",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (180, 180, 180),
            2,
        )

        # Chèn sự kiện phi cốt truyện: chữ màu đỏ "BI MAT" trong 3 giây (44.0s - 47.0s)
        if anomaly_start <= t <= anomaly_end:
            cv2.rectangle(frame, (80, 100), (560, 260), (0, 0, 40), -1)
            cv2.rectangle(frame, (80, 100), (560, 260), (0, 0, 255), 6)
            cv2.putText(
                frame,
                "BI MAT",
                (155, 205),
                cv2.FONT_HERSHEY_SIMPLEX,
                3.0,
                (0, 0, 255),  # BGR: Đỏ tươi
                8,
            )

        writer.write(frame)

    writer.release()

    # Tạo file phụ đề mô phỏng cốt truyện bình thường (không hề nhắc tới chữ "BI MAT")
    srt_content = (
        "1\n"
        "00:00:00,000 --> 00:00:30,000\n"
        "Phần mở đầu giới thiệu chuyển động cơ bản của vật thể trên màn hình.\n\n"
        "2\n"
        "00:00:30,000 --> 00:01:30,000\n"
        "Vật thể hình tròn tiếp tục di chuyển đều từ trái sang phải.\n\n"
        "3\n"
        "00:01:30,000 --> 00:02:00,000\n"
        "Kết thúc quá trình quan sát chuyển động đều.\n"
    )
    with open(srt_path, "w", encoding="utf-8") as f:
        f.write(srt_content)

    print(f"Đã tạo video kiểm thử: {video_path} ({duration_sec}s @ {fps} FPS)")
    print(f"Sự kiện chữ đỏ 'BI MAT' được chèn tại: {anomaly_start}s - {anomaly_end}s")


create_synthetic_test_video()

## Cell 5: Khởi Chạy `VideoSeekAgent` & Kiểm Chứng `QueryProfile` + `focus(fps=8)`
Chạy `VideoSeekAgent` với câu hỏi tìm dòng chữ màu đỏ xuất hiện chớp nhoáng trong video, hiển thị:
1. Kết quả phân tích câu hỏi từ `QueryAnalyzer` (`QueryProfile`).
2. Toàn bộ các bước suy luận `Thought -> Action -> Observation`.
3. Kiểm chứng Agent có sử dụng `focus` với `fps=8` (hoặc `fps >= 4`) và trả lời đúng `"BI MAT"`.

In [ ]:
import json
from videoseek.agent import VideoSeekAgent

test_question = (
    "Trong video có một dòng chữ in hoa màu đỏ xuất hiện chớp nhoáng trong khoảng 3 giây "
    "(không nằm trong phụ đề). Nội dung chính xác của dòng chữ màu đỏ đó là gì?"
)

agent = VideoSeekAgent(
    config=runtime_config,
    video_path=SYNTHETIC_VIDEO_PATH,
    subtitle_path=SYNTHETIC_SRT_PATH,
    output_dir="./outputs",
    tools=["overview", "skim", "focus"],
    verbose=True,
)

# Chạy Agent trên câu hỏi kiểm thử
trajectory = agent.run(test_question)

print("\n================ KẾT QUẢ KIỂM CHỨNG MODULE 2 ================")
if agent.query_profile is not None:
    print("1. QueryProfile từ LLM JSON Router:")
    for k, v in agent.query_profile.to_dict().items():
        print(f"   - {k}: {v}")

print("\n2. Tóm tắt Trajectory (Thought -> Action -> Observation):")
focus_fps_used = []
for step in trajectory.steps:
    fn_name = step.action.function_name
    params = step.action.parameters
    if fn_name == "focus":
        focus_fps_used.append(params.get("fps"))
    print(f"   [Step {step.step_id}] Action: {fn_name} | Params: {params}")

print("\n3. Kiểm chứng Dynamic FPS Focus & Clean Trajectory JSON:")
print(f"   - Các giá trị FPS đã truyền vào tool `focus`: {focus_fps_used}")
if any(fps == 8 for fps in focus_fps_used if fps is not None):
    print("   -> ĐẠT: Agent đã chủ động gọi `focus` với fps=8 cho sự kiện chữ chớp nhoáng!")
else:
    print("   -> Lưu ý: Kiểm tra lại log trajectory phía trên.")

traj_json_str = json.dumps(trajectory.to_dict(), ensure_ascii=False, indent=2)
print(f"   - Xuất JSON Trajectory thành công ({len(traj_json_str)} bytes, không rò rỉ non-serializable objects).")

print(f"\n4. Final Answer: {trajectory.final_answer}")
print(f"   Finish Reason: {trajectory.finish_reason}")